In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install packages
!pip install -q timm peft einops wandb python-dotenv


## Path Configurations
Configure your Google Drive paths below so the notebook can find your datasets and backup checkpoints dynamically.

In [ ]:
# ── PATH CONFIGURATIONS ──────────────────────────────────────────────────
# Edit these paths to point to your folders in Google Drive
GDRIVE_SANPO_REAL = "/content/drive/MyDrive/sanpo_real"
GDRIVE_SCENEFLOW   = "/content/drive/MyDrive/sceneflow"
GDRIVE_CHECKPOINTS = "/content/drive/MyDrive/WAFT-Stereo-Checkpoints"


import os
import wandb

# Setup checkpoint base directory
os.makedirs(GDRIVE_CHECKPOINTS, exist_ok=True)
print(f"Trained weights and checkpoints will be backed up directly to Google Drive: {GDRIVE_CHECKPOINTS}")

# Setup Weights & Biases login if desired
# wandb.login()


In [ ]:
import os
import wandb

# Setup checkpoint base directory
os.makedirs(GDRIVE_CHECKPOINTS, exist_ok=True)
print(f"Trained weights and checkpoints will be backed up directly to Google Drive: {GDRIVE_CHECKPOINTS}")

# Setup Weights & Biases login if desired
# wandb.login()


## 2. Preflight Check
Verify GPU hardware availability, CUDA libraries, and python dependencies.

In [ ]:
import torch
import os
import sys

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device name:", torch.cuda.get_device_name(0))
    print("Allocated CUDA memory:", torch.cuda.memory_allocated(0))
else:
    print("WARNING: No GPU detected. Training will be extremely slow or fail on CPU!")

## 3. Dataset Availability Check
Verify directories and file formats for both SceneFlow and Sanpo-Real datasets.

In [ ]:
import glob
import os

# Create symlinks from mounted Google Drive to the local workspace
os.makedirs("datasets", exist_ok=True)

if os.path.exists(GDRIVE_SANPO_REAL):
    print(f"Creating symlink for sanpo_real dataset from: {GDRIVE_SANPO_REAL}")
    if os.path.exists("datasets/sanpo_real"):
        if os.path.islink("datasets/sanpo_real"):
            os.remove("datasets/sanpo_real")
        else:
            import shutil
            shutil.rmtree("datasets/sanpo_real")
    os.symlink(GDRIVE_SANPO_REAL, "datasets/sanpo_real")

if os.path.exists(GDRIVE_SCENEFLOW):
    print(f"Creating symlink for sceneflow dataset from: {GDRIVE_SCENEFLOW}")
    if os.path.exists("datasets/sceneflow"):
        if os.path.islink("datasets/sceneflow"):
            os.remove("datasets/sceneflow")
        else:
            import shutil
            shutil.rmtree("datasets/sceneflow")
    os.symlink(GDRIVE_SCENEFLOW, "datasets/sceneflow")

def check_dataset(name, root):
    print(f"Checking dataset '{name}' in path: {root}...")
    if not os.path.exists(root):
        print(f"  [MISSING] Folder {root} does not exist!")
        return
    
    if name == "sceneflow":
        things_path = os.path.join(root, 'FlyingThings3D')
        driving_path = os.path.join(root, 'driving')
        things_exists = os.path.exists(things_path)
        driving_exists = os.path.exists(driving_path)
        print(f"  FlyingThings3D exists: {things_exists}")
        print(f"  Driving exists: {driving_exists}")
        if things_exists:
            left_imgs = glob.glob(os.path.join(things_path, 'frames_cleanpass/TRAIN/*/*/left/*.png'))
            disparities = glob.glob(os.path.join(things_path, 'disparity/TRAIN/*/*/left/*.pfm'))
            print(f"    FlyingThings3D TRAIN Left Images: {len(left_imgs)}")
            print(f"    FlyingThings3D TRAIN Disparities: {len(disparities)}")
        if driving_exists:
            left_imgs = glob.glob(os.path.join(driving_path, 'frames_cleanpass/*/*/*/left/*.png'))
            disparities = glob.glob(os.path.join(driving_path, 'disparity/*/*/*/left/*.pfm'))
            print(f"    Driving Left Images: {len(left_imgs)}")
            print(f"    Driving Disparities: {len(disparities)}")
    else:
        sessions = glob.glob(os.path.join(root, 'session_*'))
        print(f"  Found {len(sessions)} session folders.")
        if len(sessions) > 0:
            first_sess = sorted(sessions)[0]
            left_imgs = glob.glob(os.path.join(first_sess, 'left', '*.png'))
            right_imgs = glob.glob(os.path.join(first_sess, 'right', '*.png'))
            depths = glob.glob(os.path.join(first_sess, 'depth_ml', '*.npy')) + glob.glob(os.path.join(first_sess, 'depth_ml', '*.npz'))
            calib_file = os.path.join(first_sess, 'calib.json')
            
            print(f"  First Session statistics ({os.path.basename(first_sess)}):")
            print(f"    Left Images: {len(left_imgs)}")
            print(f"    Right Images: {len(right_imgs)}")
            print(f"    Depth Maps: {len(depths)}")
            print(f"    Calib JSON file exists: {os.path.exists(calib_file)}")
            
            if len(left_imgs) == 0 or len(depths) == 0 or not os.path.exists(calib_file):
                print("    [ERROR] Missing expected files inside session!")
            else:
                print("    [SUCCESS] Layout verified successfully.")

# Adjust paths if mapping to Google Drive inputs
check_dataset("sceneflow", "datasets/sceneflow")
check_dataset("sanpo_real", "datasets/sanpo_real")

import os
import wandb

# Setup checkpoint base directory
os.makedirs(GDRIVE_CHECKPOINTS, exist_ok=True)
print(f"Trained weights and checkpoints will be backed up directly to Google Drive: {GDRIVE_CHECKPOINTS}")

# Setup Weights & Biases login if desired
# wandb.login()


In [ ]:
import os
import wandb

# Setup checkpoint base directory
os.makedirs(GDRIVE_CHECKPOINTS, exist_ok=True)
print(f"Trained weights and checkpoints will be backed up directly to Google Drive: {GDRIVE_CHECKPOINTS}")

# Setup Weights & Biases login if desired
# wandb.login()


## 5. Dry-run / Preflight Pipeline Check (CPU Mode)
Before launching actual training with GPU (which consumes active GPU quota), we can perform a short **dry-run** on CPU. 
This dry-run initializes the MobileNetV4 model, builds the training dataset loaders, passes a single batch through the network, computes losses, performs backpropagation, updates weights, and checks checkpoint saves. Setting `SOLVER.MAX_ITER=5` ensures it completes in seconds without GPU resources.

In [ ]:
import subprocess
import time
import os
from IPython.display import display, Image, clear_output

# Setup dry-run checkpoints directory dynamically
small_ckpt_dir = os.path.join(GDRIVE_CHECKPOINTS, "custom/stage1-sceneflow-small/42")

# Run dry-run for 5 steps on CPU, with EVAL_PERIOD = 1 to trigger validation and visualization immediately
cmd = (
    "CUDA_VISIBLE_DEVICES=\"\" python main.py "
    "--config-file configs/custom/stage1-sceneflow-small.yaml "
    f"--checkpoint-dir {small_ckpt_dir} "
    "--num-gpus 0 "
    "SOLVER.MAX_ITER 5 "
    "SOLVER.IMS_PER_BATCH 1 "
    "SOLVER.CHECKPOINT_PERIOD 10 "
    "SOLVER.LATEST_CHECKPOINT_PERIOD 10 "
    "TEST.EVAL_PERIOD 0"
)

print("Starting CPU Dry-Run subprocess...")
print(f"Saving test outputs to: {small_ckpt_dir}")
process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

viz_file = os.path.join(small_ckpt_dir, "val_viz_latest.png")

try:
    while process.poll() is None:
        # Wait for the first visualization image to be generated
        if os.path.exists(viz_file):
            clear_output(wait=True)
            print("=== WAFT-Stereo Training: Dynamic Validation Visualization (Updated periodically) ===")
            display(Image(filename=viz_file))
            time.sleep(10)
        else:
            line = process.stdout.readline()
            if line:
                print(line.strip())
except KeyboardInterrupt:
    process.terminate()
    print("Training process terminated.")

process.wait()
for line in process.stdout:
    print(line.strip())
print("Dry-run completed!")

## 6. Start Training (GPU Mode)
Run the training loop on Google Colab using a GPU instance. The script will automatically download the cloud checkpoint (if available) and continue training, saving the best models to the cloud every 50 iterations when updated.

In [ ]:
import os
import subprocess
import time
from IPython.display import display, Image, clear_output

# Start training as a background subprocess
# Choose model profile config: configs/custom/stage1-sceneflow-small.yaml, -medium.yaml, or -large.yaml
config_file = "configs/custom/stage1-sceneflow-medium.yaml"
data_name = config_file.replace('\\', '/').split('/')[-2]
alg_name = config_file.replace('\\', '/').split('/')[-1].split('.')[0]
ckpt_dir = os.path.join(GDRIVE_CHECKPOINTS, data_name, alg_name, "42")

cmd = f"python main.py --config-file {config_file} --num-gpus 1 --checkpoint-dir {ckpt_dir}"

print(f"Starting training subprocess ({config_file})...")
print(f"Saving checkpoints to: {ckpt_dir}")
process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

viz_file = os.path.join(ckpt_dir, "val_viz_latest.png")

try:
    while process.poll() is None:
        # Wait for the first visualization image to be generated
        if os.path.exists(viz_file):
            clear_output(wait=True)
            print("=== WAFT-Stereo Training: Dynamic Validation Visualization (Updated periodically) ===")
            display(Image(filename=viz_file))
            time.sleep(10)
        else:
            line = process.stdout.readline()
            if line:
                print(line.strip())
except KeyboardInterrupt:
    process.terminate()
    print("Training process terminated.")

process.wait()
for line in process.stdout:
    print(line.strip())
print("Training completed!")

## 7. Evaluation
Evaluate the model against test datasets to compute EPE (End-point Error) and D1 statistics.

In [ ]:
# Evaluate using checkpoint_best.pth or latest from Google Drive
medium_ckpt_dir = os.path.join(GDRIVE_CHECKPOINTS, "custom/stage1-sceneflow-medium/42")
ckpt_path = os.path.join(medium_ckpt_dir, "checkpoint_best.pth")
cmd = f"python main.py --config-file configs/custom/stage1-sceneflow-medium.yaml --eval-only --ckpt {ckpt_path} --checkpoint-dir {medium_ckpt_dir}"

print(f"Running evaluation using checkpoint: {ckpt_path}")
!{cmd}